# Stiffness 2 — the spatial discretization

Pure transport on the same lab column as the previous notebook: convection and axial dispersion, and **nothing else**. No particle, no binding, no reaction. This is the most isolated setting in the chapter.

| | |
|---|---|
| bed length / diameter | 0.1 m / 0.01 m |
| superficial velocity | 4.2·10⁻⁴ m/s (150 cm/h) |
| bed porosity | 0.37 |
| axial dispersion | 10⁻⁷ m²/s — column Péclet ≈ 1100 |
| feed | 1 mol/m³, 60 s pulse of tracer |

Every *dissipative* process — binding, film transfer, pore diffusion, dispersion — contributes a real, negative eigenvalue, and the previous notebook showed those are free. Convection is the one operator that does not: discretized, its eigenvalues reach **off the real axis**, towards the imaginary one. If stiffness is ever going to constrain CADET's integrator, this is where it has to happen.

So the knob is the **polynomial degree, swept at a fixed 160 degrees of freedom** — P1 on 80 elements, P3 on 40, P15 on 10. The system size is constant, the physics is constant, and only the shape of the spectrum changes.

In [ ]:
# This notebook lives one folder below the workshop root, while `utility`
# and everything under `data/` are addressed relative to that root.
# Step up into it before importing anything.
import os
import sys

root = os.path.abspath(os.curdir)
while not os.path.isdir(os.path.join(root, 'utility')):
    root, previous = os.path.dirname(root), root
    if root == previous:
        raise RuntimeError('no folder containing utility/ above this notebook')

os.chdir(root)
if root not in sys.path:
    sys.path.insert(0, root)

In [ ]:
import utility.transport_spectrum_explorer as explorer

explorer.show_setting()

## Why this notebook has two knobs

One knob cannot answer the question. A small time step has two possible causes, and they look identical in a step count:

- **accuracy** — the integrator *chooses* small steps to meet your tolerance;
- **stability** — the integrator *is forced* into small steps regardless of what you asked for.

They are told apart by asking for less accuracy. Loosen the tolerance and an accuracy-limited step size relaxes in proportion; a stability-limited one cannot move, because the constraint was never about accuracy.

So the second slider is the **relative tolerance**. Sweep the polynomial degree to change the spectrum, sweep the tolerance to find out which of the two you are paying for.

Watch three columns: the **stiffness indicator** (rate of convection across one grid node, over the slowest process in the model), the **mean BDF order**, and the **step count**. A stability constraint announces itself by pushing the order down — BDF reaches for a smaller, more stable order when eigenvalues leave the region it can handle.

In [ ]:
from ipywidgets import interact

show_transport = explorer.make_explorer()

interact(show_transport, **show_transport.sliders);

## What happened

**The order never moves.** Across every polynomial degree, at every tolerance, the mean BDF order sits at 4.99 and more than 99 % of steps are taken at order 5. The integrator is never reaching for stability.

**The spectrum grows and nothing happens.** From P3 to P15 the stiffness indicator rises from 4·10⁵ to 2.6·10⁶ while the step count goes 3139 → 3433, **+9 %**.

**And the steps that are taken are all bought back by tolerance.** Loosening the tolerance by four decades cuts the step count to **0.24×** — in every configuration, including the most extreme. A stability-limited step size could not be relieved that way. These steps are taken for accuracy.

So pure transport behaves like everything else in this chapter: **CADET's time integrator is not constrained by the spectrum of the spatial discretization.** It spends its steps resolving the solution.

<details>
<summary><b>Click to learn more: pushing it to the point of absurdity</b></summary>

The realistic column has a Péclet number of about 1100. Cutting the dispersion until the Péclet number reaches $1.1\cdot10^{9}$ — essentially pure advection, a case nobody would run — the step count does finally respond to the polynomial degree: P1 takes 4517 steps and P15 takes 11898, a factor of **2.6**.

But the mean BDF order there is still **4.99**, with 99.6 % of steps at order 5, and loosening the tolerance still cuts the step count to 0.23×. Even in that corner the integrator is paying for accuracy, on a pulse edge that has become nearly discontinuous and that a high order method resolves more sharply. It is the solution that got harder, not the system.

You can reproduce this: `explorer.make_explorer(col_dispersion=1e-13)`.

</details>

<details>
<summary><b>A prediction this notebook falsified, and a measurement it corrected</b></summary>

This study was built expecting the opposite result. BDF of order 5 is only stable within 52° of the negative real axis, so a convective spectrum reaching towards the imaginary axis should eventually force the order down. It does not, at any degree or Péclet number we can reach.

An earlier measurement in this chapter seemed to support the expectation: on a two component Langmuir case, going from P3 to P8 dropped the share of steps taken at order 5 from 99 % to 88 %. That comparison was made at a **fixed element count**, so the P8 run also had 2.25 times the degrees of freedom, and the case contained a self-sharpening front. At fixed degrees of freedom and without a front, the effect disappears entirely — it was the front being resolved, not the spectrum.

The honest summary is that we have not found the mechanism by which the spatial spectrum would cost anything in a realistic chromatography model, and we have looked in the place where it should have been easiest to find.

</details>

> **Take-away.** Choosing a higher polynomial degree changes what your discretization costs *per step*, and changes how well it resolves your solution. It does not change what the time integrator has to do about stability. If a high order run is slow, look at the degrees of freedom and at the sharpness of what you are resolving — not at the spectrum.